# Chapter 6
Use the `qiskit` environment described in the repository README. Restart the kernel and run all cells in order. The code cells reproduce the final book listings.


In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import real_amplitudes, zz_feature_map
from qiskit_machine_learning.neural_networks import EstimatorQNN, SamplerQNN

num_qubits = 2

feature_map = zz_feature_map(
    feature_dimension=num_qubits,
    reps=1,
    entanglement="linear",
)
ansatz = real_amplitudes(
    num_qubits=num_qubits,
    reps=2,
    entanglement="linear",
)

circuit = QuantumCircuit(num_qubits)
circuit.compose(feature_map, inplace=True)
circuit.compose(ansatz, inplace=True)

estimator_qnn = EstimatorQNN(
    circuit=circuit,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters,
)

def parity(bitstring_as_int):
    return bin(bitstring_as_int).count("1") % 2

sampler_qnn = SamplerQNN(
    circuit=circuit,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters,
    interpret=parity,
    output_shape=2,
)


In [ ]:
import numpy as np
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.algorithms import QSVC
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.primitives import QMLSampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y_raw = make_classification(
    n_samples=24,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_repeated=0,
    n_clusters_per_class=1,
    class_sep=1.3,
    random_state=7,
)
y = np.where(y_raw == 0, -1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=7
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Angle-like feature maps are periodic. Keep inputs in a controlled range.
X_train_angles = np.clip(X_train_scaled, -np.pi, np.pi)
X_test_angles = np.clip(X_test_scaled, -np.pi, np.pi)

feature_map = zz_feature_map(
    feature_dimension=2,
    reps=2,
    entanglement="linear",
)

fidelity = ComputeUncompute(sampler=QMLSampler(shots=None))
quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map,
    fidelity=fidelity,
    enforce_psd=True,
    evaluate_duplicates="off_diagonal",
)

qsvc = QSVC(quantum_kernel=quantum_kernel, C=1.0)
qsvc.fit(X_train_angles, y_train)
qsvc_predictions = qsvc.predict(X_test_angles)

rbf_svc = SVC(kernel="rbf", C=1.0, gamma="scale")
rbf_svc.fit(X_train_scaled, y_train)
rbf_predictions = rbf_svc.predict(X_test_scaled)

print("QSVC accuracy:", accuracy_score(y_test, qsvc_predictions))
print("RBF SVC accuracy:", accuracy_score(y_test, rbf_predictions))


In [ ]:
import numpy as np

K_train = quantum_kernel.evaluate(X_train_angles)
K_test = quantum_kernel.evaluate(X_test_angles, X_train_angles)

symmetry_error = np.max(np.abs(K_train - K_train.T))
eigenvalues = np.linalg.eigvalsh((K_train + K_train.T) / 2.0)
min_eigenvalue = np.min(eigenvalues)

off_diagonal = K_train[~np.eye(K_train.shape[0], dtype=bool)]
summary = {
    "shape": K_train.shape,
    "test_shape": K_test.shape,
    "diagonal_min": float(np.min(np.diag(K_train))),
    "diagonal_max": float(np.max(np.diag(K_train))),
    "symmetry_error": float(symmetry_error),
    "min_eigenvalue": float(min_eigenvalue),
    "off_diag_mean": float(np.mean(off_diagonal)),
    "off_diag_std": float(np.std(off_diagonal)),
}

for key, value in summary.items():
    print(f"{key:>16}: {value}")


In [ ]:
from math import pi
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector

def paired_feature_map():
    """Return a two-qubit circuit with local and pairwise input phases."""
    x = ParameterVector("x", length=2)
    circuit = QuantumCircuit(2, name="paired_feature_map")

    # Local periodic features.
    circuit.h(0)
    circuit.h(1)
    circuit.rz(x[0], 0)
    circuit.rz(x[1], 1)

    # A pairwise interaction feature.
    circuit.cx(0, 1)
    circuit.rz((pi - x[0]) * (pi - x[1]), 1)
    circuit.cx(0, 1)

    return circuit

feature_map = paired_feature_map()
print(feature_map.draw(output="text", fold=68))


In [ ]:
from qiskit.circuit.library import zz_feature_map
from qiskit.transpiler import CouplingMap, generate_preset_pass_manager

feature_map = zz_feature_map(
    feature_dimension=3,
    reps=2,
    entanglement="full",
)

line_coupling = CouplingMap.from_line(3)
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    basis_gates=["rz", "sx", "x", "cx"],
    coupling_map=line_coupling,
    seed_transpiler=11,
)

transpiled = pass_manager.run(feature_map)

def circuit_report(label, circuit):
    print(f"\n{label}")
    print("depth:", circuit.depth())
    print("size:", circuit.size())
    print("ops:", dict(circuit.count_ops()))

circuit_report("original", feature_map)
circuit_report("transpiled", transpiled)


In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.001, 1),
    ["rz", "sx", "x"],
)
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.01, 2),
    ["cx"],
)

circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()

backend = AerSimulator(
    noise_model=noise_model,
    seed_simulator=123,
)
compiled = transpile(
    circuit,
    backend,
    seed_transpiler=123,
)
result = backend.run(compiled, shots=2000).result()
print(result.get_counts())


In [ ]:
import numpy as np
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import LinearSVC, SVC

def trig_features(X):
    return np.concatenate(
        [X, np.sin(X), np.cos(X), np.sin(X[:, :1] - X[:, 1:2])],
        axis=1,
    )

models = {
    "linear_svc": make_pipeline(
        StandardScaler(),
        LinearSVC(C=1.0, random_state=7),
    ),
    "rbf_svc": make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", C=1.0, gamma="scale"),
    ),
    "trig_linear_svc": make_pipeline(
        StandardScaler(),
        FunctionTransformer(trig_features, validate=False),
        StandardScaler(),
        LinearSVC(C=1.0, random_state=7),
    ),
    "qsvc": qsvc,
}

results = []
for name, model in models.items():
    if name == "qsvc":
        model.fit(X_train_angles, y_train)
        predictions = model.predict(X_test_angles)
    else:
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

    results.append({
        "model": name,
        "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
        "macro_f1": f1_score(y_test, predictions, average="macro"),
    })

for row in results:
    print(row)
